# **Tahap 5 - Feature Engineering**

## Import Library

In [22]:
import numpy as np
import pandas as pd
from pathlib import Path

## Path Konfigurasi

In [23]:
BASE_DIR     = Path(__file__).resolve().parent.parent \
               if "__file__" in dir() else Path.cwd().parent
 
FEATURES_DIR = BASE_DIR / "data" / "features"
 
if not FEATURES_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/features tidak ditemukan: {FEATURES_DIR}\n"
        f"Jalankan dulu: python src/04_preprocessing.py"
    )

## Konstanta

In [24]:
TRAIN_YEARS = list(range(2015, 2022))
TEST_YEARS  = list(range(2022, 2025))

DANGER_BINS   = [0, 100, 200, 350, float('inf')]
DANGER_LABELS = ['Rendah', 'Sedang', 'Tinggi', 'Berbahaya']
DANGER_ENC    = {'Rendah': 0, 'Sedang': 1, 'Tinggi': 2, 'Berbahaya': 3}

## Fitur Temporal (lag, rolling, YoY)

In [25]:
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['provinsi', 'tahun'])
    df['crime_rate_lag1'] = df.groupby('provinsi')['crime_rate_per100k'].shift(1)
    df['crime_rate_lag2'] = df.groupby('provinsi')['crime_rate_per100k'].shift(2)
    return df
 
 
def add_rolling_mean(df: pd.DataFrame, window: int = 3) -> pd.DataFrame:
    df = df.sort_values(['provinsi', 'tahun'])
    df['crime_rate_ma3'] = (
        df.groupby('provinsi')['crime_rate_per100k']
        .transform(lambda x: x.rolling(window=window, min_periods=2).mean())
        .round(1)
    )
    return df
 
def add_yoy_change(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['provinsi', 'tahun'])
    lag1 = df['crime_rate_lag1']
 
    df['crime_rate_yoy_abs']    = (df['crime_rate_per100k'] - lag1).round(1)
    df['crime_rate_yoy_change'] = np.where(
        lag1.notna() & (lag1 != 0),
        ((df['crime_rate_per100k'] - lag1) / lag1 * 100).round(1),
        np.nan
    )
    return df
 
 
def add_trend_direction(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['provinsi', 'tahun'])
    ma3_lag = df.groupby('provinsi')['crime_rate_ma3'].shift(1)
 
    def classify(row):
        cur, prev = row['crime_rate_ma3'], row['_ma3_lag']
        if pd.isna(cur) or pd.isna(prev) or prev == 0:
            return np.nan
        ratio = cur / prev
        if ratio > 1.05:
            return 'Naik'
        elif ratio < 0.95:
            return 'Turun'
        return 'Stabil'
 
    df['_ma3_lag']         = ma3_lag
    df['trend_direction']  = df.apply(classify, axis=1)
    df['trend_direction_enc'] = df['trend_direction'].map({'Naik': 1, 'Stabil': 0, 'Turun': -1})
    df = df.drop(columns=['_ma3_lag'])
    return df

## Fitur Interaksi

In [26]:
def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
 
    if 'unemployment_rate' in df.columns and 'population_density' in df.columns:
        df['unemp_x_density'] = (
            df['unemployment_rate'] * df['population_density']
        ).round(2)
 
    if 'crime_rate_per100k' in df.columns and 'population_density' in df.columns:
        df['crime_per_density'] = np.where(
            df['population_density'] > 0,
            (df['crime_rate_per100k'] / df['population_density']).round(4),
            np.nan
        )
 
    return df
 

## Target Valuable

In [27]:
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['danger_level'] = pd.cut(
        df['crime_rate_per100k'],
        bins=DANGER_BINS,
        labels=DANGER_LABELS,
        right=False
    )
    df['danger_level_enc'] = df['danger_level'].map(DANGER_ENC)
    return df

## Validation

In [28]:
def print_feature_summary(df: pd.DataFrame) -> pd.DataFrame:
    NEW_FEATURES = [
        'crime_rate_lag1', 'crime_rate_lag2', 'crime_rate_ma3',
        'crime_rate_yoy_abs', 'crime_rate_yoy_change',
        'trend_direction', 'trend_direction_enc',
        'unemp_x_density', 'crime_per_density',
        'danger_level', 'danger_level_enc',
    ]
    available = [f for f in NEW_FEATURES if f in df.columns]
 
    records = []
    for feat in available:
        col = df[feat]
        non_null = col.notna().sum()
        records.append({
            'feature'  : feat,
            'dtype'    : str(col.dtype),
            'non_null' : non_null,
            'null'     : col.isna().sum(),
            'null_pct' : f"{col.isna().mean()*100:.1f}%",
            'min'      : col.min() if col.dtype != 'object' and str(col.dtype) != 'category' else '-',
            'max'      : col.max() if col.dtype != 'object' and str(col.dtype) != 'category' else '-',
            'mean'     : round(col.mean(), 2) if col.dtype not in ['object'] and str(col.dtype) != 'category' else '-',
        })
 
    summary_df = pd.DataFrame(records)
    print("\n  Ringkasan fitur baru:")
    print(summary_df.to_string(index=False))
    return summary_df
 
 
def print_target_distribution(df: pd.DataFrame) -> None:
    print("\n  Distribusi danger_level (seluruh data):")
    dist = df['danger_level'].value_counts().reindex(DANGER_LABELS).fillna(0)
    total = len(df.dropna(subset=['danger_level']))
    for label, count in dist.items():
        bar = '█' * int(count / total * 30)
        print(f"    {label:<12} {int(count):>3}  {bar}  ({count/total*100:.1f}%)")
 
    print("\n  Distribusi per split:")
    for name, years in [("Train", TRAIN_YEARS), ("Test", TEST_YEARS)]:
        subset = df[df['tahun'].isin(years)]['danger_level'].value_counts()
        subset = subset.reindex(DANGER_LABELS).fillna(0).astype(int)
        print(f"    {name}: {dict(subset)}")

## Main

In [29]:
def main():
    print("\n" + "=" * 60)
    print("TAHAP 5 — FEATURE ENGINEERING")
    print(f"Input/Output : {FEATURES_DIR}")
    print("=" * 60)

    df = pd.read_csv(FEATURES_DIR / "ml_dataset.csv")
    df['tahun'] = df['tahun'].astype(int)
    print(f"\nLoaded ml_dataset.csv : {df.shape[0]} rows × {df.shape[1]} kolom")
 
    print("\n--- Membuat Fitur Temporal ---")
    df = add_lag_features(df)
    print(f"  ✅ crime_rate_lag1 & lag2")
 
    df = add_rolling_mean(df)
    print(f"  ✅ crime_rate_ma3 (window=3)")
 
    df = add_yoy_change(df)
    print(f"  ✅ crime_rate_yoy_abs & yoy_change (%)")
 
    df = add_trend_direction(df)
    print(f"  ✅ trend_direction (Naik/Stabil/Turun) + encoded")
 
    print("\n--- Membuat Fitur Interaksi ---")
    df = add_interaction_features(df)
    print(f"  ✅ unemp_x_density")
    print(f"  ✅ crime_per_density")
 
    print("\n--- Membuat Target Variable ---")
    df = add_target(df)
    print(f"  ✅ danger_level → {dict(df['danger_level'].value_counts())}")
    print(f"  ✅ danger_level_enc (0–3)")

    print("\n" + "=" * 60)
    print("RINGKASAN FITUR BARU")
    print("=" * 60)
    summary_df = print_feature_summary(df)
    print_target_distribution(df)

    print("\n--- Simpan Output ---")

    df.to_csv(FEATURES_DIR / "features_final.csv", index=False)
    print(f"  ✅ features_final.csv       → {df.shape[0]} rows × {df.shape[1]} kolom")

    train_df = df[df['tahun'].isin(TRAIN_YEARS)]
    test_df  = df[df['tahun'].isin(TEST_YEARS)]
    train_df.to_csv(FEATURES_DIR / "features_train.csv", index=False)
    test_df.to_csv( FEATURES_DIR / "features_test.csv",  index=False)
    print(f"  ✅ features_train.csv       → {len(train_df)} rows")
    print(f"  ✅ features_test.csv        → {len(test_df)} rows")
 
    summary_df.to_csv(FEATURES_DIR / "feature_summary.csv", index=False)
    print(f"  ✅ feature_summary.csv      → dokumentasi semua fitur")
 
    print(f"\nTahap 5 selesai. Lanjut ke: src/06_modeling.py")
 
 
if __name__ == "__main__":
    main()


TAHAP 5 — FEATURE ENGINEERING
Input/Output : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/features

Loaded ml_dataset.csv : 331 rows × 17 kolom

--- Membuat Fitur Temporal ---
  ✅ crime_rate_lag1 & lag2
  ✅ crime_rate_ma3 (window=3)
  ✅ crime_rate_yoy_abs & yoy_change (%)
  ✅ trend_direction (Naik/Stabil/Turun) + encoded

--- Membuat Fitur Interaksi ---
  ✅ unemp_x_density
  ✅ crime_per_density

--- Membuat Target Variable ---
  ✅ danger_level → {'Sedang': 162, 'Tinggi': 93, 'Rendah': 61, 'Berbahaya': 15}
  ✅ danger_level_enc (0–3)

RINGKASAN FITUR BARU

  Ringkasan fitur baru:
              feature    dtype  non_null  null null_pct    min      max     mean
      crime_rate_lag1  float64       297    34    10.3%    0.0    819.2   167.86
      crime_rate_lag2  float64       263    68    20.5%    0.0    446.2   158.39
       crime_rate_ma3  float64       297    34    10.3%    7.0    615.9   168.68
   crime_rate_yoy_abs  float64       297    34    10.3% -211.8    840.8